In [1]:
# One-click installation of required libraries
!pip install langchain_community
!pip install langchain-huggingface
!pip install rank-bm25
!pip install pypdf
!pip install faiss-cpu
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is 

In [2]:
# Import required dependencies
import gradio as gr
import os
from pathlib import Path
import sys
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_models import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
import logging
import re
from typing import List, Tuple, Any, Dict
import numpy as np
from langchain_core.documents import Document
import time
from datetime import datetime
import json
import pandas as pd
import concurrent.futures
from rank_bm25 import BM25Okapi

# ===========================
# Dependency Compatibility for Retry Mechanism
# ===========================
try:
    from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
    TENACITY_AVAILABLE = True
except ImportError:
    TENACITY_AVAILABLE = False

    def retry(*args, **kwargs):
        def decorator(func):
            return func
        return decorator

# ===========================
# Configuration Parameters
# ===========================
# General Parameters
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
VECTOR_STORE_PATH = "faiss_index_colab"
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_MODEL = "deepseek-chat"
TEMPERATURE = 0.1
MAX_HISTORY_ROUNDS = 5

# Multi-Strategy Retrieval Parameters
CHUNK_SIZE_CODE1 = 1000
CHUNK_OVERLAP_CODE1 = 200
ABSTRACT_WEIGHT = 0.3
SMALL_TO_BIG_WEIGHT = 0.4
SENTENCE_WINDOW_WEIGHT = 0.3
BM25_RETRIEVER_WEIGHT_CODE1 = 0.2

# Industrial Pipeline Parameters
CHUNK_SIZE_CODE2 = 800
CHUNK_OVERLAP_CODE2 = 150
RERANK_MODEL_NAME = "BAAI/bge-reranker-base"
CANDIDATE_K = 30
FINAL_K = 5
RRF_K = 60
ENABLE_QUERY_REWRITE = True

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ========== Global Variables ==========
DOCUMENTS_CHUNKS = []
BM25_INDEX = None
PARAGRAPH_CHUNKS = []
SENTENCE_CHUNKS = []
SMALL_CHUNKS = []

# Cached objects to avoid repeated loading/indexing
GLOBAL_EMBEDDINGS = None
GLOBAL_RERANKER = None
ABSTRACT_RETRIEVER = None
SMALL2BIG_RETRIEVER = None
SENTENCE_RETRIEVER = None

# ==========================================
# Global Cache Utilities
# ==========================================
def get_embeddings():
    """
    Singleton loader for Hugging Face embedding model.
    Ensures the embedding model is loaded only once globally.
    Returns:
        HuggingFaceEmbeddings: Initialized embedding model instance
    """
    global GLOBAL_EMBEDDINGS
    if GLOBAL_EMBEDDINGS is None:
        logger.info(f"Loading embeddings once: {EMBEDDING_MODEL_NAME}")
        GLOBAL_EMBEDDINGS = HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODEL_NAME,
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )
    return GLOBAL_EMBEDDINGS

def get_reranker():
    """
    Singleton loader for cross-encoder reranker model.
    Ensures the reranker model is loaded only once globally.
    Returns:
        CrossEncoder: Initialized reranker model instance
    """
    global GLOBAL_RERANKER
    if GLOBAL_RERANKER is None:
        from sentence_transformers import CrossEncoder
        logger.info(f"Loading reranker once: {RERANK_MODEL_NAME}")
        GLOBAL_RERANKER = CrossEncoder(RERANK_MODEL_NAME)
    return GLOBAL_RERANKER

# ==========================================
# Optimized RAG Generator to Resolve History Compression Errors
# ==========================================
class OptimizedRAGGenerator:
    """
    Enhanced RAG answer generation module with advanced features:
    chat history compression, iterative answer refinement, self-critique,
    and structured JSON output.
    """
    def __init__(self, llm, enable_history_compress=False, enable_refine_chain=False, enable_self_critique=False, enable_structured_output=False):
        self.llm = llm
        self.enable_history_compress = enable_history_compress
        self.enable_refine_chain = enable_refine_chain
        self.enable_self_critique = enable_self_critique
        self.enable_structured_output = enable_structured_output
        self.generation_stats = {}

    def _get_fast_prompt(self, has_compressed_history: bool = False):
        """
        Build standard prompt template for fast answer generation.
        Args:
            has_compressed_history: Whether compressed conversation history is used
        Returns:
            ChatPromptTemplate: Formatted prompt template
        """
        if has_compressed_history:
            return ChatPromptTemplate.from_messages([
                ("system", """You are a professional academic research assistant.
Answer the user's question based ONLY on the provided Context and Conversation Summary.
You MUST cite the source for every fact in this format: [File: filename.pdf | Page: X]
Do not add any information not in the context. If no relevant information, say so clearly.

Conversation Summary: {compressed_history}
Reference Context: {context}"""),
                ("human", "{question}")
            ])
        else:
            return ChatPromptTemplate.from_messages([
                ("system", """You are a professional academic research assistant.
Answer the user's question based ONLY on the provided Context.
You MUST cite the source for every fact in this format: [File: filename.pdf | Page: X]
Do not add any information not in the context. If no relevant information, say so clearly.

Context: {context}"""),
                MessagesPlaceholder(variable_name="chat_history"),
                ("human", "{question}")
            ])

    def _get_structured_prompt(self, has_compressed_history: bool = False):
        """
        Build prompt template for structured JSON output.
        Args:
            has_compressed_history: Whether compressed conversation history is used
        Returns:
            ChatPromptTemplate: JSON-formatted prompt template
        """
        base_system = """You are a professional academic research assistant. You must output JSON in the following format:
{
    "thinking": "Your step-by-step reasoning",
    "answer": "The final answer with [File | Page] citations",
    "confidence": 0.0-1.0
}
Answer based ONLY on the Context below:"""
        if has_compressed_history:
            base_system += "\nConversation Summary: {compressed_history}"
        base_system += "\nReference Context: {context}"

        if has_compressed_history:
            return ChatPromptTemplate.from_messages([
                ("system", base_system),
                ("human", "{question}")
            ])
        else:
            return ChatPromptTemplate.from_messages([
                ("system", base_system),
                MessagesPlaceholder(variable_name="chat_history"),
                ("human", "{question}")
            ])

    def _compress_history(self, chat_history: List[Tuple[str, str]]) -> Tuple[str, bool]:
        """
        Compress long chat history into concise summary to reduce token usage.
        Args:
            chat_history: List of (user_query, assistant_response) pairs
        Returns:
            Tuple: Compressed history text and success flag
        """
        if not chat_history or len(chat_history) < 2:
            history_text = "\n".join([f"User: {q}\nAssistant: {a}" for q, a in chat_history])
            return history_text if history_text else "No prior conversation.", True

        history_text = "\n".join([f"User: {q}\nAssistant: {a}" for q, a in chat_history])
        prompt = f"""Summarize the following conversation into a concise memory (max 200 words).
Keep only key entities, core facts and user's research focus.
Conversation:
{history_text}
Summary:"""

        try:
            response = self.llm.invoke([HumanMessage(content=prompt)])
            compressed = response.content.strip()
            return compressed, True
        except Exception as e:
            logger.warning(f"History compression failed: {e}")
            return history_text, False

    def _refine_answer_loop(self, query: str, docs: List[str], compressed_history: str = "") -> Dict:
        """
        Iteratively refine answer using multiple reference documents.
        Args:
            query: User question
            docs: List of formatted reference documents
            compressed_history: Compressed chat history
        Returns:
            Dict: Final refined answer and reasoning state
        """
        state = {"answer": "", "thinking": "Initial draft"}
        for i, doc in enumerate(docs):
            if i == 0:
                prompt = f"""Conversation Summary: {compressed_history}
Reference Document: {doc}
User Question: {query}
Write an initial answer with [File | Page] citations."""
                response = self.llm.invoke([HumanMessage(content=prompt)])
                state["answer"] = response.content
            else:
                prompt = f"""Current Answer: {state['answer']}
New Reference Document: {doc}
Update the answer if new document has better information, keep citations."""
                response = self.llm.invoke([HumanMessage(content=prompt)])
                state["answer"] = response.content
        return state

    def _self_critique_check(self, answer: str, full_context: str) -> Tuple[str, bool]:
        """
        Self-verify answer for hallucinations, citation accuracy, and context alignment.
        Args:
            answer: Generated answer
            full_context: Retrieved reference context
        Returns:
            Tuple: Check result and pass/fail flag
        """
        prompt = f"""Proposed Answer: {answer}
Full Context: {full_context}
Is this answer fully supported by the context? No hallucination? Correct citations?
Reply with only "PASS" or "FAIL" + 1 sentence reason."""
        try:
            response = self.llm.invoke([HumanMessage(content=prompt)]).content.strip()
            return response, "PASS" in response
        except Exception:
            return "Check skipped", True

    @retry(
        stop=stop_after_attempt(2),
        wait=wait_exponential(multiplier=1, min=1, max=5),
        retry=retry_if_exception_type((Exception,)) if TENACITY_AVAILABLE else lambda e: False
    )
    def _llm_invoke_with_retry(self, messages):
        """
        Wrapper for LLM invocation with retry mechanism for robustness.
        Args:
            messages: Formatted prompt messages
        Returns:
            LLM response object
        """
        return self.llm.invoke(messages)

    def generate(self, query: str, docs: List[Document], chat_history: List[Tuple[str, str]]) -> Tuple[str, Dict]:
        """
        Main generation pipeline: process history, generate answer, run post-processing.
        Args:
            query: User question
            docs: Retrieved document chunks
            chat_history: Conversation history
        Returns:
            Tuple: Final answer and generation statistics
        """
        start_time = time.time()
        stats = {"steps": ["Fast mode: single LLM call"], "errors": []}
        compressed_history_text = ""

        t0 = time.time()
        if self.enable_history_compress:
            compressed_history_text, history_compress_success = self._compress_history(chat_history)
            stats['history_compression_time'] = round(time.time() - t0, 2)
            stats['steps'].append("History Compressed" if history_compress_success else "History Compress Fallback")
        else:
            chat_history_formatted = chat_history[-MAX_HISTORY_ROUNDS:]
        stats['history_process_time'] = round(time.time() - t0, 2)

        formatted_docs = [
            f"[File: {d.metadata.get('source_file', 'N/A')} | Page: {d.metadata.get('page', 0) + 1}]\n{d.page_content}"
            for d in docs
        ]
        full_context = "\n\n".join(formatted_docs)
        stats['docs_count'] = len(formatted_docs)

        t1 = time.time()
        if self.enable_refine_chain:
            stats['steps'].append("Refine chain enabled")
            state = self._refine_answer_loop(query, formatted_docs, compressed_history_text)
            answer = state["answer"]
        else:
            if self.enable_structured_output:
                chat_prompt = self._get_structured_prompt(has_compressed_history=self.enable_history_compress)
            else:
                chat_prompt = self._get_fast_prompt(has_compressed_history=self.enable_history_compress)

            if self.enable_history_compress:
                final_prompt = chat_prompt.format_messages(
                    compressed_history=compressed_history_text,
                    context=full_context,
                    question=query
                )
            else:
                history_messages = []
                for q, a in chat_history_formatted:
                    history_messages.append(HumanMessage(content=q))
                    history_messages.append(AIMessage(content=a))
                final_prompt = chat_prompt.format_messages(
                    context=full_context,
                    chat_history=history_messages,
                    question=query
                )

            try:
                response = self._llm_invoke_with_retry(final_prompt)
                if self.enable_structured_output:
                    json_str = re.search(r'\{[\s\S]*\}', response.content)
                    if json_str:
                        parsed = json.loads(json_str.group(0))
                        answer = parsed.get("answer", response.content)
                        stats['thinking'] = parsed.get("thinking", "")
                    else:
                        answer = response.content
                else:
                    answer = response.content.strip()
            except Exception as e:
                stats['errors'].append(str(e))
                answer = f"❌ Generation error: {str(e)}"

        stats['generation_core_time'] = round(time.time() - t1, 2)

        if self.enable_self_critique:
            t2 = time.time()
            critique_msg, passed = self._self_critique_check(answer, full_context)
            stats['critique'] = critique_msg
            stats['passed_check'] = passed
            stats['critique_time'] = round(time.time() - t2, 2)
            stats['steps'].append("Self-critique completed")

        stats['total_gen_time'] = round(time.time() - start_time, 2)
        return answer, stats

# ==========================================
# Retrieval Layer Core Components
# ==========================================
class BM25RetrieverWrapper:
    """
    Wrapper for BM25 sparse retrieval algorithm.
    Implements keyword-based document matching.
    """
    def __init__(self, bm25_index, documents):
        self.bm25_index = bm25_index
        self.documents = documents

    def retrieve(self, query: str, k: int = 10) -> List[Tuple[Any, float]]:
        """
        Retrieve top-k relevant documents using BM25 scoring.
        Args:
            query: User search query
            k: Number of top results to return
        Returns:
            List: (document, score) pairs sorted by relevance
        """
        try:
            query_tokens = query.lower().split()
            scores = self.bm25_index.get_scores(query_tokens)
            top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
            results = [(self.documents[i], scores[i]) for i in top_indices if scores[i] > 0]
            return results
        except Exception as e:
            logger.error(f"BM25 retrieve error: {str(e)}")
            return []

def normalize_scores(scores: List[float]) -> List[float]:
    """
    Normalize retrieval scores to [0,1] range for fusion.
    Args:
        scores: Raw similarity scores
    Returns:
        List: Normalized scores
    """
    if not scores:
        return []
    scores_np = np.array(scores)
    min_score = np.min(scores_np)
    max_score = np.max(scores_np)
    if max_score == min_score:
        return [1.0] * len(scores)
    return ((scores_np - min_score) / (max_score - min_score)).tolist()

class AbstractRetrieval:
    """
    Page-level abstract retrieval strategy: match query with page abstracts first,
    then return full page content.
    """
    def __init__(self, vector_store, documents):
        self.vector_store = vector_store
        self.documents = documents
        self.abstract_index = None
        self._build_abstract_index()

    def _build_abstract_index(self):
        """Build FAISS index for document page abstracts."""
        abstracts = []
        current_key = None
        page_abstract = []

        for doc in self.documents:
            source_file = doc.metadata.get('source_file')
            page_num = doc.metadata.get('page', 0)
            key = (source_file, page_num)

            if key != current_key:
                if page_abstract and current_key is not None:
                    prev_source_file, prev_page_num = current_key
                    abstract_text = " ".join(page_abstract[:3])
                    abstract_doc = Document(
                        page_content=abstract_text,
                        metadata={'page': prev_page_num, 'source_file': prev_source_file, 'type': 'abstract'}
                    )
                    abstracts.append(abstract_doc)
                current_key = key
                page_abstract = []

            page_abstract.append(doc.page_content)

        if page_abstract and current_key is not None:
            prev_source_file, prev_page_num = current_key
            abstract_text = " ".join(page_abstract[:3])
            abstract_doc = Document(
                page_content=abstract_text,
                metadata={'page': prev_page_num, 'source_file': prev_source_file, 'type': 'abstract'}
            )
            abstracts.append(abstract_doc)

        if abstracts:
            self.abstract_index = FAISS.from_documents(abstracts, get_embeddings())

    def retrieve(self, query: str, k: int = 5) -> List[Tuple[Any, float]]:
        """Retrieve documents by matching query with page abstracts."""
        if not self.abstract_index:
            return []
        results = self.abstract_index.similarity_search_with_score(query, k=k)
        docs_with_scores = []
        for abstract_doc, score in results:
            page_num = abstract_doc.metadata.get('page')
            source_file = abstract_doc.metadata.get('source_file')
            full_docs = [
                d for d in self.documents
                if d.metadata.get('page', 0) == page_num and d.metadata.get('source_file') == source_file
            ]
            if full_docs:
                docs_with_scores.append((full_docs[0], 1 - score))
        return docs_with_scores

class SmallToBigRetrieval:
    """
    Small-to-big chunk retrieval: retrieve small fine-grained chunks first,
    then map back to large parent chunks for context completeness.
    """
    def __init__(self, vector_store, small_chunks, big_chunks):
        self.vector_store = vector_store
        self.small_chunks = small_chunks
        self.big_chunks = big_chunks
        self.small_index = None
        self._build_small_index()

    def _build_small_index(self):
        """Build FAISS index for small document chunks."""
        if self.small_chunks:
            self.small_index = FAISS.from_documents(self.small_chunks, get_embeddings())

    def retrieve(self, query: str, k: int = 5) -> List[Tuple[Any, float]]:
        """Retrieve parent large chunks via small chunk matching."""
        if not self.small_index:
            return []
        small_results = self.small_index.similarity_search_with_score(query, k=k * 2)
        big_docs = {}
        for d, s in small_results:
            pid = d.metadata.get('parent_id', 0)
            if 0 <= pid < len(self.big_chunks):
                bd = self.big_chunks[pid]
                if id(bd) not in big_docs or (1 - s) > big_docs[id(bd)][1]:
                    big_docs[id(bd)] = (bd, 1 - s)
        return sorted(big_docs.values(), key=lambda x: x[1], reverse=True)[:k]

class SentenceWindowRetrieval:
    """
    Sentence window retrieval: retrieve target sentences + surrounding context window
    to preserve semantic completeness.
    """
    def __init__(self, sentences, window_size=2):
        self.sentences = sentences
        self.window_size = window_size
        self.sentence_index = None
        self._build_sentence_index()

    def _build_sentence_index(self):
        """Build FAISS index for sentence-level chunks."""
        if self.sentences:
            self.sentence_index = FAISS.from_documents(self.sentences, get_embeddings())

    def retrieve(self, query: str, k: int = 5) -> List[Tuple[Any, float]]:
        """Retrieve sentence windows around matched sentences."""
        if not self.sentence_index:
            return []
        res = self.sentence_index.similarity_search_with_score(query, k=k)
        window_docs = []
        for sent_doc, score in res:
            idx = sent_doc.metadata.get('sentence_idx', 0)
            source_file = sent_doc.metadata.get('source_file')
            page = sent_doc.metadata.get('page')

            start = max(0, idx - self.window_size)
            end = min(len(self.sentences), idx + self.window_size + 1)
            window = [
                s for s in self.sentences[start:end]
                if s.metadata.get('source_file') == source_file and s.metadata.get('page') == page
            ]
            if not window:
                window = [sent_doc]

            txt = "\n".join([s.page_content for s in window])
            wd = Document(page_content=txt, metadata=sent_doc.metadata)
            window_docs.append((wd, 1 - score))
        return window_docs

class CombinedRetriever:
    """
    Weighted fusion of multiple retrieval strategies for enhanced performance.
    """
    def __init__(self, retrievers: List[Tuple[str, Any, float]]):
        self.retrievers = retrievers

    def retrieve(self, query: str, k: int = 5) -> List[Tuple[Any, float]]:
        """
        Fuse results from multiple retrievers with weighted scores.
        Args:
            query: User query
            k: Top-k results to return
        Returns:
            List: Fused and sorted (document, score) pairs
        """
        candidates = {}
        for name, ret, weight in self.retrievers:
            try:
                res = ret.retrieve(query, k=k * 2)
                if not res:
                    continue
                scores = [s for _, s in res]
                nscores = normalize_scores(scores)
                for (doc, _), ns in zip(res, nscores):
                    key = id(doc)
                    if key not in candidates:
                        candidates[key] = {'doc': doc, 'score': 0, 'sources': []}
                    candidates[key]['score'] += weight * ns
                    candidates[key]['sources'].append(name)
            except Exception:
                continue
        sorted_cand = sorted(candidates.values(), key=lambda x: x['score'], reverse=True)[:k]
        return [(c['doc'], c['score']) for c in sorted_cand]

def rewrite_query_with_history(query: str, chat_history: List[Tuple[str, str]], llm) -> Tuple[List[str], str]:
    """
    Rewrite user query using conversation history: resolve pronouns and generate sub-queries.
    Args:
        query: Original user question
        chat_history: Conversation history
        llm: LLM for query rewriting
    Returns:
        Tuple: List of sub-queries and core entity
    """
    rewrite_prompt = """You are a professional query rewriting assistant for academic paper retrieval.
Your tasks:
1. **Entity Resolution**: Identify all pronouns (it/this model/this method etc.) in the current question and replace them with the specific full name mentioned in the chat history.
2. **Query Generation**: Generate 2-3 retrieval-friendly sub-queries.
3. **Format**: Output the core entity first, then a blank line, then the sub-queries.

Now process:
Chat History:
{chat_history}
Current Question: {query}
Output:"""
    try:
        history_text = ""
        for q, a in chat_history[-MAX_HISTORY_ROUNDS:]:
            history_text += f"User: {q}\nAssistant: {a}\n"
        response = llm.invoke(rewrite_prompt.format(chat_history=history_text, query=query))
        content = response.content.strip()
        lines = [l.strip() for l in content.split("\n") if l.strip()]
        core_entity = lines[0] if lines else ""
        sub_queries = lines[1:] if len(lines) > 1 else [query]
        if core_entity:
            verified_queries = []
            for q in sub_queries:
                if core_entity.lower() not in q.lower():
                    verified_queries.append(f"{core_entity} {q}")
                else:
                    verified_queries.append(q)
            sub_queries = verified_queries[:3]
        return sub_queries, core_entity
    except Exception as e:
        logger.warning(f"Query rewrite failed: {str(e)}")
        return [query], ""

def build_bm25_index(chunks: List[Any]):
    """
    Build BM25 index from document chunks for sparse retrieval.
    Args:
        chunks: List of document chunks
    Returns:
        BM25Okapi: Initialized BM25 index
    """
    corpus = [chunk.page_content.lower().split() for chunk in chunks]
    return BM25Okapi(corpus)

def bm25_search(query: str, k: int = CANDIDATE_K) -> List[Tuple[Any, float]]:
    """
    Perform BM25 search on global document chunks.
    Args:
        query: Search query
        k: Number of top results
    Returns:
        List: (document, score) pairs
    """
    if not BM25_INDEX or not DOCUMENTS_CHUNKS:
        return []
    try:
        query_tokens = query.lower().split()
        scores = BM25_INDEX.get_scores(query_tokens)
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
        return [(DOCUMENTS_CHUNKS[i], scores[i]) for i in top_indices if scores[i] > 0]
    except Exception as e:
        logger.error(f"BM25 retrieve error: {str(e)}")
        return []

def compute_rrf(dense_results: List[Tuple[Any, float]], sparse_results: List[Tuple[Any, float]], k: int = RRF_K) -> List[Tuple[Any, float]]:
    """
    Reciprocal Rank Fusion (RRF) for dense and sparse retrieval results.
    Args:
        dense_results: Dense embedding retrieval results
        sparse_results: BM25 sparse retrieval results
        k: RRF ranking constant
    Returns:
        List: Fused and sorted results
    """
    rrf_scores = {}
    for rank, (doc, score) in enumerate(dense_results):
        doc_id = id(doc)
        if doc_id not in rrf_scores:
            rrf_scores[doc_id] = {"doc": doc, "score": 0.0}
        rrf_scores[doc_id]["score"] += 1.0 / (k + rank + 1)
    for rank, (doc, score) in enumerate(sparse_results):
        doc_id = id(doc)
        if doc_id not in rrf_scores:
            rrf_scores[doc_id] = {"doc": doc, "score": 0.0}
        rrf_scores[doc_id]["score"] += 1.0 / (k + rank + 1)
    sorted_docs = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)
    return [(item["doc"], item["score"]) for item in sorted_docs]

def rerank_docs(query: str, docs_with_scores: List[Tuple[Any, float]], top_k: int = FINAL_K) -> List[Tuple[Any, float]]:
    """
    Rerank candidate documents using cross-encoder model for higher accuracy.
    Args:
        query: User query
        docs_with_scores: Candidate documents with initial scores
        top_k: Number of final reranked results
    Returns:
        List: Reranked (document, score) pairs
    """
    try:
        if not docs_with_scores:
            return []
        reranker = get_reranker()
        docs = [d for d, _ in docs_with_scores]
        pairs = [[query, doc.page_content] for doc in docs]
        scores = reranker.predict(pairs)
        scored_docs = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
        return scored_docs[:top_k]
    except Exception as e:
        logger.error(f"Rerank failed: {str(e)}")
        return sorted(docs_with_scores, key=lambda x: x[1], reverse=True)[:top_k]

def analyze_and_route_query(query: str, chat_history: List[Tuple[str, str]], available_files: List[str], llm) -> List[dict]:
    """
    Query routing: split complex query into sub-queries and map to relevant files.
    Args:
        query: User question
        chat_history: Conversation history
        available_files: List of uploaded PDF files
        llm: LLM for query analysis
    Returns:
        List: Query routing plans with sub-queries and target files
    """
    if not available_files or not llm:
        return [{"query": query, "files": ["ALL"]}]
    history_text = ""
    for q, a in chat_history[-MAX_HISTORY_ROUNDS:]:
        history_text += f"User: {q}\nAssistant: {a}\n"

    prompt = f"""You are an expert academic retrieval router.
Available Files: {json.dumps(available_files, indent=2)}
Chat History: {history_text}
Current Question: {query}
Task: Split into sub-queries and map to files. Output JSON array only."""

    try:
        response = llm.invoke(prompt).content.strip()
        json_str = re.search(r'\[.*\]', response, re.DOTALL)
        if json_str:
            plans = json.loads(json_str.group(0))
            valid_plans = [p for p in plans if "query" in p and "files" in p]
            if valid_plans:
                return valid_plans
    except Exception as e:
        logger.warning(f"Routing failed: {str(e)}")
    return [{"query": query, "files": ["ALL"]}]

# ==========================================
# General Utility Functions
# ==========================================
def clean_text(text):
    """
    Clean raw text: remove extra whitespaces and redundant punctuation.
    Args:
        text: Raw input text
    Returns:
        str: Cleaned text
    """
    if not text:
        return ""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\.{2,}', '.', text)
    return text.strip()

def setup_llm(api_key):
    """
    Initialize and validate DeepSeek LLM connection.
    Args:
        api_key: DeepSeek API key
    Returns:
        Tuple: LLM instance and status message
    """
    if not api_key or api_key.strip() == "":
        return None, "❌ Please enter DeepSeek API Key"
    try:
        llm = ChatOpenAI(
            model=DEEPSEEK_MODEL,
            api_key=api_key.strip(),
            base_url=DEEPSEEK_BASE_URL,
            temperature=TEMPERATURE,
            max_tokens=4096,
            request_timeout=30
        )
        return llm, "✅ API Key verified successfully"
    except Exception as e:
        return None, f"❌ API setup error: {str(e)}"

def format_monitoring_output(data: dict) -> str:
    """
    Format retrieval/generation monitoring data into human-readable text.
    Args:
        data: Monitoring statistics dictionary
    Returns:
        str: Formatted monitoring log
    """
    strategy = data.get('retrieval_strategy', 'unknown')
    docs = data.get('num_docs_retrieved', 0)
    total = round(data.get('total_time', 0), 3)

    gen_info = ""
    if 'gen_stats' in data:
        gs = data['gen_stats']
        gen_info = f"\n⚙️ Gen Layer Steps: {gs.get('steps', [])}"
        if 'total_gen_time' in gs:
            gen_info += f" | Gen Time: {gs['total_gen_time']}s"
        if gs.get('errors'):
            gen_info += f"\n⚠️ Errors: {gs['errors']}"

    info = f"""🔍 Retrieval Monitor:
- Strategy Used: {strategy}
- Docs Retrieved: {docs}
- Total Time: {total}s{gen_info}"""
    if data.get('error'):
        info += f"\n- ⚠️ Error: {data.get('error')}"
    return info

def build_retriever_cache(vector_store):
    """
    Pre-build and cache all multi-strategy retrievers for fast access.
    Args:
        vector_store: FAISS vector store instance
    """
    global ABSTRACT_RETRIEVER, SMALL2BIG_RETRIEVER, SENTENCE_RETRIEVER
    try:
        ABSTRACT_RETRIEVER = AbstractRetrieval(vector_store, PARAGRAPH_CHUNKS) if PARAGRAPH_CHUNKS else None
    except Exception as e:
        logger.warning(f"Build AbstractRetrieval failed: {e}")
        ABSTRACT_RETRIEVER = None

    try:
        SMALL2BIG_RETRIEVER = SmallToBigRetrieval(vector_store, SMALL_CHUNKS, PARAGRAPH_CHUNKS) if SMALL_CHUNKS and PARAGRAPH_CHUNKS else None
    except Exception as e:
        logger.warning(f"Build SmallToBigRetrieval failed: {e}")
        SMALL2BIG_RETRIEVER = None

    try:
        SENTENCE_RETRIEVER = SentenceWindowRetrieval(SENTENCE_CHUNKS, window_size=2) if SENTENCE_CHUNKS else None
    except Exception as e:
        logger.warning(f"Build SentenceWindowRetrieval failed: {e}")
        SENTENCE_RETRIEVER = None

def _guess_parent_id_for_small_chunk(small_doc, big_chunks_by_page):
    """
    Map small chunks to parent large chunks by content matching and page location.
    Args:
        small_doc: Small document chunk
        big_chunks_by_page: Large chunks grouped by page
    Returns:
        int: Parent chunk index
    """
    key = (small_doc.metadata.get("source_file"), small_doc.metadata.get("page"))
    candidate_indices = big_chunks_by_page.get(key, [])
    if not candidate_indices:
        return 0

    small_text = small_doc.page_content.strip()
    small_anchor = small_text[:60] if small_text else ""

    for idx in candidate_indices:
        big_text = PARAGRAPH_CHUNKS[idx].page_content
        if small_anchor and small_anchor in big_text:
            return idx

    return candidate_indices[0]

def process_pdf(uploaded_files):
    """
    Process uploaded PDFs: load, split, chunk, build indexes and cache data.
    Args:
        uploaded_files: List of uploaded PDF files
    Returns:
        Tuple: Vector store instance and status message
    """
    if not uploaded_files:
        return None, "❌ Please upload at least one PDF."
    try:
        all_docs = []
        for file in uploaded_files:
            loader = PyPDFLoader(file.name)
            docs = loader.load()
            file_basename = os.path.basename(file.name)
            for d in docs:
                d.page_content = clean_text(d.page_content)
                if 'page' not in d.metadata:
                    d.metadata['page'] = 0
                d.metadata['source_file'] = file_basename
            all_docs.extend(docs)

        splitter_code2 = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE_CODE2,
            chunk_overlap=CHUNK_OVERLAP_CODE2,
            separators=["\n\n", "\n", ". ", "。", "！", "？"],
            length_function=len
        )
        para_chunks_code2 = splitter_code2.split_documents(all_docs)

        splitter_code1 = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE_CODE1,
            chunk_overlap=CHUNK_OVERLAP_CODE1,
            separators=["\n\n", "\n", ". ", "。", "！", "？"],
            length_function=len
        )
        para_chunks_code1 = splitter_code1.split_documents(all_docs)

        global DOCUMENTS_CHUNKS, BM25_INDEX, PARAGRAPH_CHUNKS, SENTENCE_CHUNKS, SMALL_CHUNKS
        DOCUMENTS_CHUNKS = para_chunks_code2
        PARAGRAPH_CHUNKS = para_chunks_code1

        big_chunks_by_page = {}
        for idx, big_doc in enumerate(PARAGRAPH_CHUNKS):
            key = (big_doc.metadata.get("source_file"), big_doc.metadata.get("page"))
            big_chunks_by_page.setdefault(key, []).append(idx)

        small_split = RecursiveCharacterTextSplitter(
            chunk_size=200,
            chunk_overlap=50,
            separators=[". ", "。", "！", "？"],
            length_function=len
        )
        small_chunks = small_split.split_documents(all_docs)
        for i, s in enumerate(small_chunks):
            s.metadata['chunk_id'] = i
            s.metadata['parent_id'] = _guess_parent_id_for_small_chunk(s, big_chunks_by_page)
        SMALL_CHUNKS = small_chunks

        sent_split = RecursiveCharacterTextSplitter(
            chunk_size=100,
            chunk_overlap=20,
            separators=[". ", "。", "！", "？"],
            length_function=len
        )
        sent_chunks = sent_split.split_documents(all_docs)
        for i, s in enumerate(sent_chunks):
            s.metadata['sentence_idx'] = i
        SENTENCE_CHUNKS = sent_chunks

        BM25_INDEX = build_bm25_index(DOCUMENTS_CHUNKS)

        vs = FAISS.from_documents(DOCUMENTS_CHUNKS, get_embeddings())
        vs.save_local(VECTOR_STORE_PATH)

        import pickle
        with open(f"{VECTOR_STORE_PATH}/chunks.pkl", "wb") as f:
            pickle.dump(DOCUMENTS_CHUNKS, f)
        with open(f"{VECTOR_STORE_PATH}/chunks_code1.pkl", "wb") as f:
            pickle.dump(PARAGRAPH_CHUNKS, f)
        with open(f"{VECTOR_STORE_PATH}/small_chunks.pkl", "wb") as f:
            pickle.dump(SMALL_CHUNKS, f)
        with open(f"{VECTOR_STORE_PATH}/sent_chunks.pkl", "wb") as f:
            pickle.dump(SENTENCE_CHUNKS, f)

        build_retriever_cache(vs)

        return vs, (
            f"✅ Success!\nFiles: {len(uploaded_files)}\n"
            f"Chunks (Pipeline): {len(para_chunks_code2)}\n"
            f"Chunks (Strategy): {len(para_chunks_code1)}"
        )
    except Exception as e:
        return None, f"❌ Error: {str(e)}"

def load_vector_store():
    """
    Load saved FAISS index and cached chunks from local storage.
    Returns:
        FAISS: Loaded vector store instance
    """
    if not os.path.exists(VECTOR_STORE_PATH):
        return None
    try:
        vs = FAISS.load_local(VECTOR_STORE_PATH, get_embeddings(), allow_dangerous_deserialization=True)
        import pickle
        global DOCUMENTS_CHUNKS, BM25_INDEX, PARAGRAPH_CHUNKS, SENTENCE_CHUNKS, SMALL_CHUNKS
        if os.path.exists(f"{VECTOR_STORE_PATH}/chunks.pkl"):
            with open(f"{VECTOR_STORE_PATH}/chunks.pkl", "rb") as f:
                DOCUMENTS_CHUNKS = pickle.load(f)
            BM25_INDEX = build_bm25_index(DOCUMENTS_CHUNKS)
        if os.path.exists(f"{VECTOR_STORE_PATH}/chunks_code1.pkl"):
            with open(f"{VECTOR_STORE_PATH}/chunks_code1.pkl", "rb") as f:
                PARAGRAPH_CHUNKS = pickle.load(f)
        if os.path.exists(f"{VECTOR_STORE_PATH}/small_chunks.pkl"):
            with open(f"{VECTOR_STORE_PATH}/small_chunks.pkl", "rb") as f:
                SMALL_CHUNKS = pickle.load(f)
        if os.path.exists(f"{VECTOR_STORE_PATH}/sent_chunks.pkl"):
            with open(f"{VECTOR_STORE_PATH}/sent_chunks.pkl", "rb") as f:
                SENTENCE_CHUNKS = pickle.load(f)

        build_retriever_cache(vs)
        return vs
    except Exception as e:
        logger.warning(f"Load vector store failed: {e}")
        return None

# ==========================================
# Core RAG Q&A Engine
# ==========================================
def answer_question_with_history(
    user_question,
    chat_history: List[Tuple[str, str]],
    vector_store,
    llm,
    retrieval_mode,
    en_abs,
    en_stb,
    en_sent,
    en_bm25_code1,
    en_history_compress,
    en_refine_chain,
    en_self_critique,
    en_structured_output,
    fast_eval_mode=False
):
    """
    Main RAG pipeline: retrieve relevant docs and generate answer with conversation history.
    Supports two retrieval modes: Multi-Strategy and Industrial Pipeline.
    Returns:
        Tuple: Answer, sources, relevance score, monitor log, updated chat history
    """
    start_time = time.time()
    monitoring_data = {
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'question': user_question[:100],
        'retrieval_strategy': retrieval_mode,
        'retrieval_time': 0,
        'total_time': 0,
        'num_docs_retrieved': 0,
        'error': None,
        'gen_stats': {}
    }

    if not llm or not vector_store or not user_question.strip():
        return "❌ Please check API key, PDF and question", "", "Low", format_monitoring_output(monitoring_data), chat_history

    final_docs_with_scores = []
    retrieval_start = time.time()

    if retrieval_mode == "Multi-Strategy (Code 1)":
        try:
            if (en_abs and ABSTRACT_RETRIEVER is None) or (en_stb and SMALL2BIG_RETRIEVER is None) or (en_sent and SENTENCE_RETRIEVER is None):
                build_retriever_cache(vector_store)

            retrievers = []
            if en_abs and ABSTRACT_RETRIEVER is not None:
                retrievers.append(("abstract", ABSTRACT_RETRIEVER, ABSTRACT_WEIGHT))
            if en_stb and SMALL2BIG_RETRIEVER is not None:
                retrievers.append(("small2big", SMALL2BIG_RETRIEVER, SMALL_TO_BIG_WEIGHT))
            if en_sent and SENTENCE_RETRIEVER is not None:
                retrievers.append(("sentence", SENTENCE_RETRIEVER, SENTENCE_WINDOW_WEIGHT))

            if retrievers:
                cr = CombinedRetriever(retrievers)
                queries, core_entity = rewrite_query_with_history(user_question, chat_history, llm)

                all_candidates = []
                for q in queries:
                    results = cr.retrieve(q, k=FINAL_K * 2)
                    all_candidates.extend(results)

                unique_docs = {}
                for doc, score in all_candidates:
                    doc_id = id(doc)
                    if doc_id not in unique_docs or score > unique_docs[doc_id][1]:
                        unique_docs[doc_id] = (doc, score)

                final_docs_with_scores = sorted(unique_docs.values(), key=lambda x: x[1], reverse=True)[:FINAL_K]
            else:
                final_docs_with_scores = vector_store.similarity_search_with_score(user_question, k=FINAL_K)
                final_docs_with_scores = [(d, 1 - s) for d, s in final_docs_with_scores]

        except Exception as e:
            monitoring_data['error'] = str(e)
            final_docs_with_scores = vector_store.similarity_search_with_score(user_question, k=FINAL_K)
            final_docs_with_scores = [(d, 1 - s) for d, s in final_docs_with_scores]

    else:
        global DOCUMENTS_CHUNKS
        available_files = list(set([
            doc.metadata.get('source_file')
            for doc in DOCUMENTS_CHUNKS
            if doc.metadata.get('source_file')
        ]))

        candidate_k = 10 if fast_eval_mode else CANDIDATE_K
        final_k = 3 if fast_eval_mode else FINAL_K
        use_query_rewrite = ENABLE_QUERY_REWRITE and (not fast_eval_mode)

        if use_query_rewrite:
            routing_plans = analyze_and_route_query(user_question, chat_history, available_files, llm)
        else:
            routing_plans = [{"query": user_question, "files": ["ALL"]}]

        all_rrf_candidates = {}
        for plan in routing_plans:
            q = plan["query"]
            targets = plan["files"]

            dense_res = []
            if "ALL" in targets or not targets:
                dense_res = vector_store.similarity_search_with_score(q, k=candidate_k)
            else:
                res = vector_store.similarity_search_with_score(q, k=candidate_k * 2)
                filtered_res = [r for r in res if r[0].metadata.get("source_file") in targets]
                dense_res.extend(filtered_res)

            sparse_res_unfiltered = bm25_search(q, k=candidate_k * 3)
            sparse_res = []
            if "ALL" in targets or not targets:
                sparse_res = sparse_res_unfiltered[:candidate_k]
            else:
                sparse_res = [r for r in sparse_res_unfiltered if r[0].metadata.get("source_file") in targets][:candidate_k]

            rrf_res = compute_rrf(dense_res, sparse_res, k=RRF_K)
            for doc, score in rrf_res:
                doc_id = id(doc)
                if doc_id not in all_rrf_candidates or score > all_rrf_candidates[doc_id][1]:
                    all_rrf_candidates[doc_id] = (doc, score)

        sorted_cand = sorted(all_rrf_candidates.values(), key=lambda x: x[1], reverse=True)[:int(candidate_k * 1.5)]
        final_docs_with_scores = rerank_docs(user_question, sorted_cand, top_k=final_k)

    monitoring_data['retrieval_time'] = time.time() - retrieval_start
    monitoring_data['num_docs_retrieved'] = len(final_docs_with_scores)

    if not final_docs_with_scores:
        new_history = chat_history + [(user_question, "No relevant information found.")]
        return "No relevant information found.", "", "Low", format_monitoring_output(monitoring_data), new_history

    docs_list = [d for d, _ in final_docs_with_scores]

    generator = OptimizedRAGGenerator(
        llm,
        enable_history_compress=en_history_compress,
        enable_refine_chain=en_refine_chain,
        enable_self_critique=en_self_critique,
        enable_structured_output=en_structured_output
    )
    answer, gen_stats = generator.generate(user_question, docs_list, chat_history)
    monitoring_data['gen_stats'] = gen_stats

    monitoring_data['total_time'] = time.time() - start_time

    sources_text = ""
    cited_pages = set()
    for idx, (doc, score) in enumerate(final_docs_with_scores):
        page_num = doc.metadata.get('page', 0) + 1
        source_file = doc.metadata.get('source_file', 'Unknown File')
        page_key = f"{source_file}_{page_num}"
        if page_key not in cited_pages:
            cited_pages.add(page_key)
            formatted_score = f"{score:.4f}" if isinstance(score, (float, int, np.floating)) else str(score)
            sources_text += (
                f"**Source {idx + 1} | File: {source_file} | Page {page_num} | Score: {formatted_score}**:\n"
                f"{doc.page_content[:350]}...\n\n"
            )

    similarity = "High" if len(cited_pages) > 0 else "Low"
    new_history = chat_history + [(user_question, answer)]
    if len(new_history) > MAX_HISTORY_ROUNDS:
        new_history = new_history[-MAX_HISTORY_ROUNDS:]

    return answer, sources_text, similarity, format_monitoring_output(monitoring_data), new_history

# ===========================
# Batch Evaluation Module
# ===========================
def safe_json_loads(text: str):
    """
    Robust JSON parsing with fallback for malformed JSON.
    Args:
        text: Input JSON string
    Returns:
        dict/list: Parsed JSON or None
    """
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        pass
    try:
        match = re.search(r'\{[\s\S]*\}', text)
        if match:
            return json.loads(match.group(0))
    except Exception:
        pass
    return None

def normalize_filename(name: str) -> str:
    """Normalize filename for consistent comparison."""
    if not name:
        return ""
    return os.path.basename(str(name)).strip().lower()

def normalize_text_simple(text: str) -> str:
    """Normalize text: lowercase, trim spaces, remove extra whitespace."""
    if text is None:
        return ""
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def parse_sources_metadata(sources_text: str) -> List[Dict[str, Any]]:
    """Parse retrieved source metadata (file, page, score) from display text."""
    results = []
    if not sources_text:
        return results
    pattern = re.compile(
        r"\*\*Source\s+\d+\s+\|\s+File:\s+(.*?)\s+\|\s+Page\s+(\d+)\s+\|\s+Score:\s+([^\*]+)\*\*",
        re.MULTILINE
    )
    for match in pattern.finditer(sources_text):
        file_name = match.group(1).strip()
        page = int(match.group(2))
        score_text = match.group(3).strip()
        try:
            score_val = float(score_text)
        except Exception:
            score_val = None
        results.append({
            "file": file_name,
            "file_norm": normalize_filename(file_name),
            "page": page,
            "score": score_val
        })
    return results

def extract_gold_evidence(item: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Extract ground-truth evidence (file, page) from evaluation dataset item."""
    evidence = item.get("gold_evidence", [])
    parsed = []

    if isinstance(evidence, list) and evidence:
        for ev in evidence:
            if not isinstance(ev, dict):
                continue
            file_name = ev.get("file", "")
            pages = ev.get("pages", [])
            if isinstance(pages, int):
                pages = [pages]
            pages = [int(p) for p in pages if isinstance(p, (int, float, str)) and str(p).isdigit()]
            if file_name:
                parsed.append({
                    "file": file_name,
                    "file_norm": normalize_filename(file_name),
                    "pages": pages
                })

    if not parsed:
        gold_files = item.get("gold_files", [])
        gold_pages = item.get("gold_pages", [])
        if isinstance(gold_files, str):
            gold_files = [gold_files]
        if isinstance(gold_pages, int):
            gold_pages = [gold_pages]
        pages = [int(p) for p in gold_pages if isinstance(p, (int, float, str)) and str(p).isdigit()]
        for gf in gold_files:
            parsed.append({
                "file": gf,
                "file_norm": normalize_filename(gf),
                "pages": pages
            })
    return parsed

def compute_retrieval_metrics(sources_text: str, item: Dict[str, Any]) -> Dict[str, Any]:
    """
    Calculate retrieval evaluation metrics: file hit, page hit, top-k accuracy.
    Args:
        sources_text: Retrieved sources display text
        item: Dataset item with ground truth
    Returns:
        Dict: Retrieval performance metrics
    """
    retrieved = parse_sources_metadata(sources_text)
    gold_evidence = extract_gold_evidence(item)

    gold_files = {ev["file_norm"] for ev in gold_evidence}
    gold_pairs = set()
    for ev in gold_evidence:
        for p in ev.get("pages", []):
            gold_pairs.add((ev["file_norm"], int(p)))

    retrieved_files = [r["file"] for r in retrieved]
    retrieved_pages = [r["page"] for r in retrieved]

    file_hit = any(r["file_norm"] in gold_files for r in retrieved) if gold_files else None
    page_hit = any((r["file_norm"], int(r["page"])) in gold_pairs for r in retrieved) if gold_pairs else None

    top1_file_hit = None
    top1_page_hit = None
    if retrieved:
        top1 = retrieved[0]
        top1_file_hit = top1["file_norm"] in gold_files if gold_files else None
        top1_page_hit = ((top1["file_norm"], int(top1["page"])) in gold_pairs) if gold_pairs else None

    return {
        "gold_evidence": gold_evidence,
        "retrieved": retrieved,
        "retrieved_files": retrieved_files,
        "retrieved_pages": retrieved_pages,
        "file_hit": file_hit,
        "page_hit": page_hit,
        "top1_file_hit": top1_file_hit,
        "top1_page_hit": top1_page_hit
    }

def has_required_citation(answer: str) -> bool:
    """Check if answer contains required [File | Page] citations."""
    if not answer:
        return False
    return re.search(r'\[File:\s*.+?\|\s*Page:\s*\d+\]', answer) is not None

def build_evaluation_prompt(item: Dict[str, Any], generated_answer: str, sources_text: str) -> str:
    """Build prompt for LLM-based answer evaluation."""
    question = item.get("question", "")
    ref_answer = item.get("reference_answer", "")
    question_type = item.get("question_type", "unknown")
    key_points = item.get("key_points", [])
    answer_keywords = item.get("answer_keywords", [])
    must_cite = item.get("must_cite", True)

    return f"""You are evaluating a RAG system answer.

Please score the answer on four dimensions:
1. correctness (0-5): Is the answer factually correct compared with the reference answer?
2. completeness (0-5): Does it cover the main expected points?
3. groundedness (0-5): Is the answer supported by the retrieved sources shown below, rather than unsupported guesswork?
4. citation_quality (0-5): Are citations present and appropriate? If must_cite=true and the answer has no explicit citations like [File: ... | Page: X], score this low.

Question Type: {question_type}
Must Cite: {must_cite}

Question:
{question}

Reference Answer:
{ref_answer}

Expected Key Points:
{json.dumps(key_points, ensure_ascii=False)}

Answer Keywords / Aliases:
{json.dumps(answer_keywords, ensure_ascii=False)}

Generated Answer:
{generated_answer}

Retrieved Sources:
{sources_text}

Return ONLY valid JSON in this exact schema:
{{
  "correctness": 0,
  "completeness": 0,
  "groundedness": 0,
  "citation_quality": 0,
  "overall": 0,
  "matched_key_points": [],
  "missing_key_points": [],
  "reason": "brief explanation"
}}"""

def parse_judge_result(eval_text: str) -> Dict[str, Any]:
    """Parse LLM evaluation JSON response into structured metrics."""
    default = {
        "correctness": 0,
        "completeness": 0,
        "groundedness": 0,
        "citation_quality": 0,
        "overall": 0,
        "matched_key_points": [],
        "missing_key_points": [],
        "reason": "Failed to parse evaluation response."
    }

    parsed = safe_json_loads(eval_text)
    if not isinstance(parsed, dict):
        return default

    result = default.copy()
    for key in ["correctness", "completeness", "groundedness", "citation_quality", "overall"]:
        try:
            result[key] = int(parsed.get(key, 0))
        except Exception:
            result[key] = 0

    matched = parsed.get("matched_key_points", [])
    missing = parsed.get("missing_key_points", [])
    result["matched_key_points"] = matched if isinstance(matched, list) else []
    result["missing_key_points"] = missing if isinstance(missing, list) else []
    result["reason"] = str(parsed.get("reason", "")).strip() or default["reason"]
    return result

def run_batch_evaluation(
    json_file_path, vector_store, llm,
    mode, en_abs, en_stb, en_sent,
    en_hc, en_ref, en_sc, en_so,
    progress=gr.Progress()
):
    """
    Run automated batch evaluation on JSON dataset.
    Compute retrieval and generation metrics, export detailed results.
    """
    if not llm:
        return None, "❌ Please verify API Key first.", None
    if not vector_store:
        return None, "❌ Please process PDFs first.", None
    if not json_file_path:
        return None, "❌ Please upload JSON dataset.", None

    try:
        with open(json_file_path.name, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
    except Exception as e:
        return None, f"❌ Failed to parse JSON: {str(e)}", None

    if not isinstance(dataset, list):
        return None, "❌ JSON dataset must be a list of question items.", None

    total_questions = len(dataset)
    if total_questions == 0:
        return None, "❌ JSON dataset is empty.", None

    results = [None] * total_questions
    completed = 0
    progress(0, desc="Starting Evaluation...")

    def evaluate_single_item(idx, item):
        question = item.get("question", "")
        if not question:
            return None

        generated_answer, sources_text, _, _, _ = answer_question_with_history(
            question,
            [],
            vector_store,
            llm,
            mode,
            en_abs,
            en_stb,
            en_sent,
            False,
            en_hc,
            en_ref,
            en_sc,
            en_so,
            fast_eval_mode=True
        )

        retrieval_metrics = compute_retrieval_metrics(sources_text, item)
        citation_present = has_required_citation(generated_answer)

        judge_prompt = build_evaluation_prompt(item, generated_answer, sources_text)

        try:
            eval_response = llm.invoke(judge_prompt).content.strip()
            judge = parse_judge_result(eval_response)
        except Exception as e:
            eval_response = json.dumps({
                "correctness": 0,
                "completeness": 0,
                "groundedness": 0,
                "citation_quality": 0,
                "overall": 0,
                "matched_key_points": [],
                "missing_key_points": [],
                "reason": f"Error: {str(e)}"
            }, ensure_ascii=False)
            judge = parse_judge_result(eval_response)

        if item.get("must_cite", True) and not citation_present:
            judge["citation_quality"] = min(judge["citation_quality"], 1)
            judge["overall"] = max(0, judge["overall"] - 5)

        gold_evidence_text = "; ".join(
            [f'{ev["file"]} pages {ev.get("pages", [])}' for ev in retrieval_metrics["gold_evidence"]]
        )

        return {
            "ID": idx + 1,
            "QuestionType": item.get("question_type", "unknown"),
            "Difficulty": item.get("difficulty", "unknown"),
            "Question": question,
            "ReferenceAnswer": item.get("reference_answer", ""),
            "GoldEvidence": gold_evidence_text,
            "RetrievedFiles": " | ".join(retrieval_metrics["retrieved_files"]),
            "RetrievedPages": " | ".join(map(str, retrieval_metrics["retrieved_pages"])),
            "Top1FileHit": retrieval_metrics["top1_file_hit"],
            "Top1PageHit": retrieval_metrics["top1_page_hit"],
            "FileHit": retrieval_metrics["file_hit"],
            "PageHit": retrieval_metrics["page_hit"],
            "CitationPresent": citation_present,
            "Correctness": judge["correctness"],
            "Completeness": judge["completeness"],
            "Groundedness": judge["groundedness"],
            "CitationQuality": judge["citation_quality"],
            "Overall": judge["overall"],
            "MatchedKeyPoints": json.dumps(judge["matched_key_points"], ensure_ascii=False),
            "MissingKeyPoints": json.dumps(judge["missing_key_points"], ensure_ascii=False),
            "JudgeReason": judge["reason"],
            "GeneratedAnswer": generated_answer,
            "SourcesText": sources_text,
            "RawJudge": eval_response
        }

    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        future_to_idx = {executor.submit(evaluate_single_item, i, item): i for i, item in enumerate(dataset)}
        for future in concurrent.futures.as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                res = future.result()
                if res:
                    results[idx] = res
            except Exception as exc:
                print(f"Item {idx} exception: {exc}")
            completed += 1
            progress(completed / total_questions, desc=f"Evaluated {completed}/{total_questions}...")

    valid_results = [r for r in results if r is not None]
    if not valid_results:
        return None, "❌ No valid evaluation results generated.", None

    df = pd.DataFrame(valid_results)

    avg_overall = float(df["Overall"].mean()) if "Overall" in df else 0.0
    avg_correctness = float(df["Correctness"].mean()) if "Correctness" in df else 0.0
    avg_groundedness = float(df["Groundedness"].mean()) if "Groundedness" in df else 0.0

    def mean_bool(col_name):
        if col_name not in df:
            return 0.0
        valid = df[col_name].dropna()
        if len(valid) == 0:
            return 0.0
        return float(valid.astype(int).mean())

    file_hit_rate = mean_bool("FileHit")
    page_hit_rate = mean_bool("PageHit")
    citation_rate = mean_bool("CitationPresent")

    stats_text = (
        f"📊 **Complete**\n"
        f"- Total: {len(valid_results)}\n"
        f"- Avg Overall: **{avg_overall:.2f}**\n"
        f"- Avg Correctness: **{avg_correctness:.2f}/5**\n"
        f"- Avg Groundedness: **{avg_groundedness:.2f}/5**\n"
        f"- File Hit Rate: **{file_hit_rate:.1%}**\n"
        f"- Page Hit Rate: **{page_hit_rate:.1%}**\n"
        f"- Citation Presence Rate: **{citation_rate:.1%}**"
    )

    output_filename = "eval_results_detailed.json"
    df.to_json(output_filename, orient="records", indent=4, force_ascii=False)
    return df, stats_text, output_filename

# ==========================================
# Gradio Web UI
# ==========================================
with gr.Blocks(title="Hybrid RAG System") as demo:
    gr.Markdown("# 🚀 Hybrid RAG: Multi-Strategy & Industrial Pipeline")
    vs_state = gr.State(None)
    llm_state = gr.State(None)
    chat_history_state = gr.State([])

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## ⚙️ Settings")
            with gr.Group():
                api_key = gr.Textbox(label="DeepSeek API Key", type="password", placeholder="sk-...")
                verify_btn = gr.Button("Verify API", variant="primary")
                api_status = gr.Textbox(label="Status", interactive=False)

            retrieval_mode = gr.Radio(
                ["Industrial Pipeline (Code 2)", "Multi-Strategy (Code 1)"],
                label="🔀 Retrieval Mode",
                value="Industrial Pipeline (Code 2)",
                info="Choose the retrieval backend"
            )

            with gr.Group(visible=False) as strategy_options:
                gr.Markdown("### 🧩 Multi-Strategy Options")
                abs_check = gr.Checkbox(label="Abstract Retrieval", value=True)
                stb_check = gr.Checkbox(label="Small-to-Big Retrieval", value=True)
                sent_check = gr.Checkbox(label="Sentence Window", value=True)

            def toggle_strategy_options(mode):
                return gr.update(visible=mode == "Multi-Strategy (Code 1)")

            retrieval_mode.change(toggle_strategy_options, inputs=retrieval_mode, outputs=strategy_options)

            with gr.Group():
                gr.Markdown("### ⚡ Generation Layer Config")
                gr.Markdown("*Default: All off (Fastest Mode)*")
                en_history_compress = gr.Checkbox(label="Enable History Compression", value=False)
                en_refine_chain = gr.Checkbox(label="Enable Refine Chain (High Quality)", value=False)
                en_self_critique = gr.Checkbox(label="Enable Self-Critique", value=False)
                en_structured_output = gr.Checkbox(label="Enable Structured JSON Output", value=False)

            with gr.Group():
                gr.Markdown("### 📂 Document KB")
                pdf = gr.File(label="Upload PDFs", file_count="multiple")
                with gr.Row():
                    proc_btn = gr.Button("Process PDFs", variant="primary")
                    clear_btn = gr.Button("Clear Docs")
                proc_status = gr.Textbox(label="Status", lines=3, interactive=False)

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("💬 Interactive Chat"):
                    question = gr.Textbox(label="Question", lines=2, placeholder="Ask a question...")
                    ask_btn = gr.Button("Ask", variant="primary", size="lg")

                    with gr.Group():
                        sim_ind = gr.Textbox(label="Relevance", interactive=False)
                        ans = gr.Textbox(label="Answer", lines=10)

                    with gr.Accordion("Sources", open=False):
                        sources = gr.Markdown("No sources")
                    with gr.Accordion("✅ Monitor", open=False):
                        monitor_out = gr.Textbox(label="Debug Info", lines=8, interactive=False)
                    with gr.Accordion("History", open=True):
                        history_display = gr.Textbox(label="Conversation", lines=6, interactive=False)
                        clear_history_btn = gr.Button("Clear History", variant="secondary", size="sm")

                with gr.TabItem("🚀 Batch Evaluation"):
                    gr.Markdown("Upload JSON for automated testing.")
                    with gr.Row():
                        eval_json_upload = gr.File(label="Upload JSON Dataset")
                        start_eval_btn = gr.Button("▶️ Start Evaluation", variant="primary")
                    eval_stats = gr.Markdown("Status: Ready")
                    eval_results_df = gr.Dataframe(label="Results")
                    download_results = gr.File(label="Download")

    verify_btn.click(setup_llm, inputs=api_key, outputs=[llm_state, api_status])
    proc_btn.click(process_pdf, inputs=pdf, outputs=[vs_state, proc_status])

    def on_clear_docs():
        global DOCUMENTS_CHUNKS, BM25_INDEX, PARAGRAPH_CHUNKS, SENTENCE_CHUNKS, SMALL_CHUNKS
        global ABSTRACT_RETRIEVER, SMALL2BIG_RETRIEVER, SENTENCE_RETRIEVER
        DOCUMENTS_CHUNKS = []
        PARAGRAPH_CHUNKS = []
        SENTENCE_CHUNKS = []
        SMALL_CHUNKS = []
        BM25_INDEX = None
        ABSTRACT_RETRIEVER = None
        SMALL2BIG_RETRIEVER = None
        SENTENCE_RETRIEVER = None
        if os.path.exists(VECTOR_STORE_PATH):
            import shutil
            try:
                shutil.rmtree(VECTOR_STORE_PATH)
            except Exception:
                pass
        return None, "📭 Cleared.", []

    clear_btn.click(fn=on_clear_docs, outputs=[vs_state, proc_status, chat_history_state])

    def on_ask(q, vs, llm, chat_history, mode, abs_chk, stb_chk, sent_chk, en_hc, en_ref, en_sc, en_so):
        a, s, sim, mon, new_history = answer_question_with_history(
            q,
            chat_history,
            vs,
            llm,
            mode,
            abs_chk,
            stb_chk,
            sent_chk,
            False,
            en_hc,
            en_ref,
            en_sc,
            en_so
        )
        history_text = ""
        for idx, (user_q, assistant_a) in enumerate(new_history):
            history_text += f"[Round {idx + 1}]\nQ: {user_q}\nA: {assistant_a}\n\n"
        return a, s, sim, mon, history_text, new_history, ""

    ask_btn.click(
        fn=on_ask,
        inputs=[
            question,
            vs_state,
            llm_state,
            chat_history_state,
            retrieval_mode,
            abs_check,
            stb_check,
            sent_check,
            en_history_compress,
            en_refine_chain,
            en_self_critique,
            en_structured_output
        ],
        outputs=[ans, sources, sim_ind, monitor_out, history_display, chat_history_state, question]
    )

    clear_history_btn.click(fn=lambda: ("", []), outputs=[history_display, chat_history_state])

    start_eval_btn.click(
        fn=run_batch_evaluation,
        inputs=[
            eval_json_upload, vs_state, llm_state,
            retrieval_mode, abs_check, stb_check, sent_check,
            en_history_compress, en_refine_chain, en_self_critique, en_structured_output
        ],
        outputs=[eval_results_df, eval_stats, download_results]
    )

    demo.load(load_vector_store, outputs=vs_state)

if __name__ == "__main__":
    gr.close_all()
    demo.queue().launch(inbrowser=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://82c0064e936d523373.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
